In [0]:
%sql
-- Step 1: Create the schema
CREATE SCHEMA IF NOT EXISTS workspace.mspp1000;

-- Step 2: Create the volume inside that schema
CREATE VOLUME IF NOT EXISTS workspace.mspp1000.rawdata;
--SHOW CATALOGS;


In [0]:
%python
# Set up variables for clarity
catalog = "workspace"
schema = "mspp1000"
volume = "rawdata"
file_name = "D3_g.csv"

# Build the full path
file_path = f"/Volumes/{catalog}/{schema}/{volume}/{file_name}"

# Read the CSV
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Show the data
display(df)

In [0]:
%python
# Set up variables for clarity
catalog = "workspace"
schema = "mspp1000"
volume = "rawdata"
file_name = "D3_g.txt"

# Build the full path
file_path = f"/Volumes/{catalog}/{schema}/{volume}/{file_name}"

# Read the CSV
df_dim = spark.read.csv(file_path, header=False, inferSchema=True,  sep=";")

# Show the data
display(df)

# Save it as a permanent table in your schema
df_dim.write.mode("overwrite").saveAsTable("workspace.mspp1000.D3_g_dim_silver")

In [0]:
# Set up variables
catalog = "workspace"
schema = "mspp1000"
volume = "rawdata"
file_name = "D3_g.csv"
file_path = f"/Volumes/{catalog}/{schema}/{volume}/{file_name}"

# Read the CSV
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Display column names to see the actual names
print("Column names:")
print(df.columns)

# Filter for 2025 data
from pyspark.sql.functions import col

df_2025 = df.filter(col("Date").startswith("2025"))

print("2025 data only:")
display(df_2025)

# Get all column names except "Date" (these will become accounts)
account_columns = [c for c in df_2025.columns if c != "Date"]

# Escape column names with backticks for columns that have dots
from pyspark.sql.functions import expr

# Build the stack expression with escaped column names
stack_parts = []
for c in account_columns:
    # If column name has special characters, wrap in backticks
    if '.' in c or ' ' in c or '-' in c:
        escaped_c = f"`{c}`"
    else:
        escaped_c = c
    stack_parts.append(f'"{c}", {escaped_c}')

stack_expr = f"stack({len(account_columns)}, {', '.join(stack_parts)})"

df_melted = df_2025.select(
    expr("'2025'").alias("fiscal_year"),
    expr(stack_expr).alias("account", "value")
)

print("Final result - rolled up 2025 with accounts as rows:")
display(df_melted)

# Show only the columns you want, excluding "date" account
final_df = df_melted.select("fiscal_year", "account", "value") \
                   .filter(col("account") != "date")

display(final_df)

# After you have your final_df
# Save it as a permanent table in your schema
final_df.write.mode("overwrite").saveAsTable("workspace.mspp1000.D3_g_aggr_silver")

In [0]:
%sql 
CREATE OR REPLACE TABLE workspace.mspp1000.D3_g__2025_gold AS
WITH joined_data AS (
    SELECT 
        fact.fiscal_year,
        fact.account,
        fact.value,
        SPLIT(dim._c0, '\t')[1] AS account_description
    FROM workspace.mspp1000.D3_g_aggr_silver AS fact
    LEFT JOIN workspace.mspp1000.D3_g_dim_silver AS dim 
        ON fact.account = LEFT(dim._c0, 13)
)
SELECT 
    fiscal_year,
    account,
    account_description,
    SUM(value) AS total_value,
    AVG(value) AS avg_value,
    COUNT(*) AS record_count
FROM joined_data
GROUP BY 
    fiscal_year,
    account,
    account_description
ORDER BY account

In [0]:
%sql
select * from workspace.mspp1000.D3_g__2025_gold
